In [255]:
import os
import numpy as np
import flopy
import matplotlib.pyplot as plt

In [256]:
##############################
### random well generation ###
##############################

# number of wells
n_wells = 3

# time steps for pumping rate changes
t_eval_val = 365.0 # [day]
dt = 7 # [days]
time_steps = np.arange(dt, t_eval_val + dt, dt)
if time_steps[-1] > t_eval_val: time_steps[-1] = t_eval_val

# bounds of coordinate system
x_min, x_max = -50, 50
y_min, y_max = -50, 50

# wells cannot be within this many units of the centre (0,0)
no_go_radius = 10.0

# range of pumping rates [m3/day]
q_min, q_max = 6.5, 65

wells = []
while len(wells) < n_wells:
    wx, wy = np.random.uniform(x_min, x_max), np.random.uniform(y_min, y_max)
    # check distance for all wells from centre
    if np.sqrt(wx**2 + wy**2) > no_go_radius:
        wells.append({
            'X': wx, 'Y': wy, 
            'Q_series': np.random.uniform(q_min, q_max, len(time_steps))
        })

In [257]:
import os
import numpy as np
import flopy
import math
from scipy.special import exp1
import plotly.graph_objects as go

# --- 1. Randomized Domain and Square Cell Logic (Canadian English) ---

# 1a. Randomise square cell size (e.g., between 1.5m and 4.0m)
cell_dim = float(np.random.uniform(1.5, 4.0))

# 1b. Randomise target domain size (e.g., between 80m and 150m)
target_lx = np.random.uniform(80, 150)
target_ly = np.random.uniform(80, 150)

# 1c. Snap domain to cell size to ensure integer number of square cells
ncol = int(target_lx / cell_dim)
nrow = int(target_ly / cell_dim)
lx = float(ncol * cell_dim)
ly = float(nrow * cell_dim)

# 1d. Adjust radii to fit the randomized rectangular domain
compliance_radius = float(min(lx, ly) * 0.42) 
no_go_radius = float(compliance_radius * 0.3)
n_wells = 3
num_cp = 12

# --- 2. Initialize MODFLOW 6 Model ---
model_name = "robust_rect_model"
workspace = "mf6_work"
if not os.path.exists(workspace):
    os.makedirs(workspace)

sim = flopy.mf6.MFSimulation(sim_name=model_name, exe_name="mf6", sim_ws=workspace)
tdis = flopy.mf6.ModflowTdis(sim, nper=1, perioddata=[(365.0, 1, 1.0)], time_units="DAYS")
gwf = flopy.mf6.ModflowGwf(sim, modelname=model_name, save_flows=True)
ims = flopy.mf6.ModflowIms(sim, complexity="SIMPLE")

# Centring logic for a rectangular domain using xorigin/yorigin
dis = flopy.mf6.ModflowGwfdis(
    gwf, nlay=1, nrow=nrow, ncol=ncol, 
    delr=cell_dim, delc=cell_dim, 
    top=0.0, botm=-10.0,
    xorigin=-lx/2, yorigin=-ly/2
)

# --- 3. Generate Random Wells ---
wells = []
while len(wells) < n_wells:
    wx = float(np.random.uniform(-lx/2, lx/2))
    wy = float(np.random.uniform(-ly/2, ly/2))
    if np.sqrt(wx**2 + wy**2) > no_go_radius:
        # Get cell indices safely
        try:
            row, col = gwf.modelgrid.intersect(wx, wy)
            wells.append({'x': wx, 'y': wy, 'row': row, 'col': col})
        except:
            continue

# --- 4. Manual Grid Line Construction (Forcing Python Floats) ---
x_edge_coords = np.linspace(-lx/2, lx/2, ncol + 1)
y_edge_coords = np.linspace(-ly/2, ly/2, nrow + 1)

grid_x, grid_y = [], []
# Vertical lines
for x in x_edge_coords:
    grid_x.extend([float(x), float(x), None])
    grid_y.extend([float(y_edge_coords[0]), float(y_edge_coords[-1]), None])
# Horizontal lines
for y in y_edge_coords:
    grid_x.extend([float(x_edge_coords[0]), float(x_edge_coords[-1]), None])
    grid_y.extend([float(y), float(y), None])

# --- 5. Visualization ---
fig = go.Figure()

# MODFLOW Grid (The "Silver Mesh")
fig.add_trace(go.Scatter(
    x=grid_x, y=grid_y, 
    mode='lines', 
    line=dict(color='silver', width=1),
    name="MODFLOW Grid", 
    hoverinfo='skip'
))

# Compliance Boundary (The Circle)
t_fine = np.linspace(0, 2 * np.pi, 100)
fig.add_trace(go.Scatter(
    x=[float(compliance_radius * np.cos(t)) for t in t_fine], 
    y=[float(compliance_radius * np.sin(t)) for t in t_fine],
    mode='lines', line=dict(color='blue', dash='dash', width=2), 
    name="Compliance Ring"
))

# Compliance Points (The Stars)
thetas = np.linspace(0, 2 * np.pi, num_cp, endpoint=False)
fig.add_trace(go.Scatter(
    x=[float(compliance_radius * np.cos(t)) for t in thetas], 
    y=[float(compliance_radius * np.sin(t)) for t in thetas],
    mode='markers', 
    marker=dict(color='yellow', size=12, symbol='star', line=dict(width=1, color='black')), 
    name="Compliance Points"
))

# Random Wells (The Red Dots)
fig.add_trace(go.Scatter(
    x=[w['x'] for w in wells], 
    y=[w['y'] for w in wells], 
    mode='markers', 
    marker=dict(color='red', size=10), 
    name="Random Wells"
))

# Centre Point (The Black X)
fig.add_trace(go.Scatter(
    x=[0], y=[0], 
    mode='markers', 
    marker=dict(color='black', size=12, symbol='x'), 
    name="Centre"
))

# --- 6. Final Layout & Axis Scaling ---
# Calculate a margin so the grid isn't touching the edge of the plot
plot_limit = max(lx, ly) / 2 + 10

fig.update_layout(
    title=(f"Rectangular Domain ({lx:.1f}m x {ly:.1f}m) | Square Cells ({cell_dim:.2f}m)<br>"
           f"<sup>Discretisation: {nrow} rows by {ncol} columns</sup>"),
    xaxis_title="X Coordinate (m)",
    yaxis_title="Y Coordinate (m)",
    width=800, height=800,
    template="plotly_white",
    xaxis=dict(range=[-plot_limit, plot_limit], showgrid=False, zeroline=False),
    yaxis=dict(range=[-plot_limit, plot_limit], scaleanchor="x", scaleratio=1, showgrid=False, zeroline=False)
)

fig.show()

print(f"Final Grid: {nrow} rows x {ncol} columns")
print(f"Compliance Radius: {compliance_radius:.2f} m")


Final Grid: 49 rows x 44 columns
Compliance Radius: 33.78 m
